In [1]:
import os
import pandas as pd

if os.path.exists("CarPrice_Assignment.csv"):
  print("Found CarPrice_Assignment.csv!")
  df = pd.read_csv("CarPrice_Assignment.csv")
  print("Dataset shape:", df.shape)
else:
  print("Files in this folder:", os.listdir("."))

Found CarPrice_Assignment.csv!
Dataset shape: (205, 26)


In [2]:
import numpy as np
import pandas as pd

def verify_matrices(A, B):
  A = np.asarray(A, dtype=float)
  B = np.asarray(B, dtype=float)

  if A.shape != B.shape:
    raise ValueError(f"Shape mismatch: {A.shape} vs {B.shape}")

  C = A + B

  error = np.max(np.abs((C - A) - B))
  return C, error


class LinearRegression:

  def __init__(self):
    self.weights = None

  def fit(self, X, y):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float).reshape(-1, 1)

    ones = np.ones((X.shape[0], 1))
    X_design = np.hstack((ones, X))

    XT_X = np.dot(X_design.T, X_design)
    XT_y = np.dot(X_design.T, y)
    self.weights = np.dot(np.linalg.pinv(XT_X), XT_y)
    return self

  def predict(self, X):
    if self.weights is None:
      raise ValueError("Model is not fitted yet. Call fit() first.")
    X = np.asarray(X, dtype=float)
    ones = np.ones((X.shape[0], 1))
    X_design = np.hstack((ones, X))
    return np.dot(X_design, self.weights)

  def evaluate(self, y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float).reshape(-1, 1)
    y_pred = np.asarray(y_pred, dtype=float).reshape(-1, 1)

    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))

    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    ss_res = np.sum((y_true - y_pred) ** 2)
    r2 = 1.0 - (ss_res / ss_tot)

    return {"MAE": mae, "RMSE": rmse, "R2": r2}


mat_A = np.array([[10.0, 4.0, 6.0], [2.0, 8.0, 1.0], [5.0, 3.0, 7.0]])
mat_B = np.array([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [7.0, 8.0, 9.0]])

sum_mat, error_diff = verify_matrices(mat_A, mat_B)
print("Matrix addition verified with numerical error:", error_diff)

features = ["enginesize", "curbweight", "horsepower", "carwidth", "highwaympg"]
target = "price"

X = df[features].values
y = df[target].values

np.random.seed(42)
indices = np.arange(len(X))
np.random.shuffle(indices)

split_idx = int(0.8 * len(X))
X_train, y_train = X[indices[:split_idx]], y[indices[:split_idx]]
X_test, y_test = X[indices[split_idx:]], y[indices[split_idx:]]

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

scores = model.evaluate(y_test, y_pred)
print("\n--- Model Performance on Test Data ---")
print(f"MAE:  ${scores['MAE']:,.2f}")
print(f"RMSE: ${scores['RMSE']:,.2f}")
print(f"R²:   {scores['R2']:.4f}")

Matrix addition verified with numerical error: 0.0

--- Model Performance on Test Data ---
MAE:  $2,989.28
RMSE: $3,972.12
R²:   0.7965


In [3]:
results = pd.DataFrame({
    "Actual Price": y_test[:10],
    "Predicted Price": y_pred[:10].flatten(),
    "Difference": y_test[:10] - y_pred[:10].flatten(),
})
results.round(2)

,Actual Price,Predicted Price,Difference
0,15690.0,21946.96,-6256.96
1,15040.0,13917.37,1122.63
2,11549.0,16004.06,-4455.06
3,18420.0,19335.39,-915.39
4,10795.0,11453.81,-658.81
5,7395.0,6153.66,1241.34
6,11900.0,14269.73,-2369.73
7,5195.0,4869.83,325.17
8,19045.0,21111.10,-2066.10
9,9989.0,15382.99,-5393.99
